# Within-Subject Power Analysis (Reasoning Movement Only)

This notebook computes analytic power and required sample size for within-subject reasoning-movement metrics under:

\[
H_0: \mu_{\Delta} = 0
\]

where \(\Delta\) is each participant's change score.

Scope for this notebook:
- No simulation
- No assumption-based benchmark rates
- Reuse saved outputs from prior analyses


## Interpretation

- These are **one-sample (paired-mean style)** power calculations on participant-level change scores tested against zero.
- Observed pilot mean and SD are used to compute standardized effect size \(d = 	ext{mean}/	ext{sd}\).
- Required sample sizes are computed for target power 0.80 at alpha 0.05.
- This is a first-pass estimate; pilot effect sizes can be noisy and optimistic.
- Later, we can add assumption-based analyses for other outcomes.


In [3]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
from statsmodels.stats.power import TTestPower


## Configuration

In [4]:
# Power-analysis configuration
alpha = 0.05
target_power = 0.80
alternative = 'two-sided'  # 'two-sided', 'larger', or 'smaller'

# Primary summary path from your current notebook outputs
summary_csv_path = Path('analysis/power_calc/analysis/output/power_calc/within_subject_summary.csv')

# Optional fallback if you run from a different working directory
summary_csv_fallbacks = [
    summary_csv_path,
    Path('analysis/output/power_calc/within_subject_summary.csv'),
    Path('../analysis/output/power_calc/within_subject_summary.csv'),
]

# Optional participant-level file path (not required for this pass)
participant_csv_path = Path('analysis/power_calc/analysis/output/power_calc/within_subject_participant_metrics.csv')


## Load Existing Outputs

In [5]:
def resolve_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

resolved_summary_path = resolve_first_existing(summary_csv_fallbacks)
if resolved_summary_path is None:
    tried = '\n'.join([f' - {Path(p)}' for p in summary_csv_fallbacks])
    raise FileNotFoundError(f'Could not find within_subject_summary.csv. Tried:\n{tried}')

summary_df_raw = pd.read_csv(resolved_summary_path)
summary_df = summary_df_raw.copy()
summary_df['value_num'] = pd.to_numeric(summary_df['value'], errors='coerce')

print('Loaded summary from:', resolved_summary_path)
print('Shape:', summary_df.shape)
display(summary_df.head(20))


Loaded summary from: analysis/output/power_calc/within_subject_summary.csv
Shape: (104, 3)


,metric,value,value_num
0,n_included,18,18.000000
1,n_excluded,0,0.000000
2,top_choice_switch,0.2222222222222222,0.222222
3,top_choice_switch_n,18,18.000000
4,top_choice_switch_successes,4,4.000000
5,top_choice_switch_ci_low,0.09000928108601697,0.090009
6,top_choice_switch_ci_high,0.4521458431621262,0.452146
7,top_choice_switch_ci_method,wilson,NaN
8,any_edit,0.4444444444444444,0.444444
9,any_edit_n,18,18.000000


## Identify Target Reasoning Metrics

In [6]:
# Metric bases requested for power calculations
TARGET_METRIC_BASES = [
    'embedding_shift_intensity',
    'directional_learning_centroid',
    'directional_learning_nearest_exposed',
]


def get_summary_stat(df, metric_base, stat):
    key = f'{metric_base}_{stat}'
    row = df.loc[df['metric'] == key, 'value_num']
    if row.empty:
        return np.nan
    return float(row.iloc[0])


def build_metric_row(df, metric_base):
    return {
        'metric_name': metric_base,
        'n_observed': get_summary_stat(df, metric_base, 'n'),
        'mean_observed': get_summary_stat(df, metric_base, 'mean'),
        'sd_observed': get_summary_stat(df, metric_base, 'sd'),
    }

pilot_metrics_df = pd.DataFrame([build_metric_row(summary_df, m) for m in TARGET_METRIC_BASES])
display(pilot_metrics_df)


,metric_name,n_observed,mean_observed,sd_observed
0,embedding_shift_intensity,18.0,0.369941,0.100875
1,directional_learning_centroid,18.0,-0.061598,0.115942
2,directional_learning_nearest_exposed,18.0,-0.154386,0.133372


## Compute Effect Sizes and Power / Required N

In [8]:
power_model = TTestPower()
rows_out = []

for _, row in pilot_metrics_df.iterrows():
    metric_name = row['metric_name']
    n_obs = row['n_observed']
    mean_obs = row['mean_observed']
    sd_obs = row['sd_observed']

    notes = []

    # Validate inputs
    if pd.isna(n_obs) or pd.isna(mean_obs) or pd.isna(sd_obs):
        notes.append('missing_n_or_mean_or_sd_in_summary')
        rows_out.append({
            'metric_name': metric_name,
            'n_observed': n_obs,
            'mean_observed': mean_obs,
            'sd_observed': sd_obs,
            'effect_size_d': np.nan,
            'alpha': alpha,
            'target_power': target_power,
            'alternative': alternative,
            'required_n': np.nan,
            'achieved_power_at_observed_n': np.nan,
            'notes_or_flags': ';'.join(notes),
        })
        continue

    n_obs = float(n_obs)
    mean_obs = float(mean_obs)
    sd_obs = float(sd_obs)

    if sd_obs <= 0:
        notes.append('nonpositive_sd')
        d_signed = np.nan
        d_abs = np.nan
    else:
        d_signed = mean_obs / sd_obs
        d_abs = abs(d_signed)

    required_n = np.nan
    achieved_power = np.nan

    if pd.isna(d_abs) or d_abs == 0:
        notes.append('zero_or_nan_effect_size')
    else:
        try:
            n_star = power_model.solve_power(
                effect_size=d_abs,
                alpha=alpha,
                power=target_power,
                alternative=alternative,
            )
            required_n = int(math.ceil(n_star)) if np.isfinite(n_star) else np.nan
        except Exception as exc:
            notes.append(f'solve_power_failed:{exc}')

        try:
            if n_obs >= 2:
                achieved_power = power_model.power(
                    effect_size=d_abs,
                    nobs=n_obs,
                    alpha=alpha,
                    alternative=alternative,
                )
            else:
                notes.append('n_observed_less_than_2')
        except Exception as exc:
            notes.append(f'power_calc_failed:{exc}')

    rows_out.append({
        'metric_name': metric_name,
        'n_observed': int(n_obs) if not pd.isna(n_obs) else np.nan,
        'mean_observed': mean_obs,
        'sd_observed': sd_obs,
        'effect_size_d': d_signed,
        'alpha': alpha,
        'target_power': target_power,
        'alternative': alternative,
        'required_n': required_n,
        'achieved_power_at_observed_n': achieved_power,
        'notes_or_flags': ';'.join(notes) if notes else '',
    })

power_results_df = pd.DataFrame(rows_out)

# Pretty formatting for display only
display_cols = [
    'metric_name',
    'n_observed',
    'mean_observed',
    'sd_observed',
    'effect_size_d',
    'alpha',
    'target_power',
    'alternative',
    'required_n',
    'achieved_power_at_observed_n',
    'notes_or_flags',
]

display(power_results_df[display_cols])


,metric_name,n_observed,mean_observed,sd_observed,effect_size_d,alpha,target_power,alternative,required_n,achieved_power_at_observed_n,notes_or_flags
0,embedding_shift_intensity,18,0.369941,0.100875,3.667300,0.05,0.8,two-sided,3,NaN,
1,directional_learning_centroid,18,-0.061598,0.115942,-0.531288,0.05,0.8,two-sided,30,0.565882,
2,directional_learning_nearest_exposed,18,-0.154386,0.133372,-1.157557,0.05,0.8,two-sided,8,0.996076,


## Optional Save

In [ ]:
# Save power table next to other power-calc outputs
output_dir = Path('analysis/power_calc/analysis/output/power_calc')
output_dir.mkdir(parents=True, exist_ok=True)

power_table_out = output_dir / 'within_subject_reasoning_power_table.csv'
power_results_df.to_csv(power_table_out, index=False)

print('Saved:', power_table_out)
